In [25]:
import numpy as np
import pandas as pd

import pickle
from ARIMA import ARIMAX, plot_acf_pacf
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.stattools import adfuller

In [26]:
df = pd.read_csv("../../data/pre_data.csv")

In [27]:
exog_cols = ["Thị_trường", "Loại_giá", "Nguồn"]

In [28]:
for item in df["Tên_mặt_hàng"].unique():
    item_df = df[df["Tên_mặt_hàng"] == item]

    y = item_df["Giá"].values.tolist()
    X = item_df[exog_cols].values.tolist()

    model = ARIMAX(
        y,
        p = 1,
        d = 1,
        q = 1,
        exog=X,
    )


    with open(f"../../models/arimax/{item}.pkl", "wb") as file:
        pickle.dump(model, file)

# Chọn 1 sản phẩm

In [29]:
item_df = df[df["Tên_mặt_hàng"]==23]

In [30]:
item_df.shape

(4817, 7)

In [31]:
item_df.head()

,Tên_mặt_hàng,Thị_trường,Loại_giá,Nguồn,Ngày,Giá,Ngành_hàng
0,23,19,7,2,2020-01-01,32639.0,0
1,23,12,7,2,2020-01-01,32000.0,0
2,23,19,7,2,2020-01-02,32639.0,0
3,23,12,7,2,2020-01-02,32000.0,0
4,23,19,7,2,2020-01-03,32539.0,0


## Kiểm định adf đến khi đạt tính dừng

In [32]:
data = item_df["Giá"]

In [33]:
def find_d(y, max_d=10, significance=0.1):
    diffed = np.array(y, dtype=float)
    
    for d in range(max_d + 1):
        result = adfuller(diffed)
        p_value = result[1]
        print(f"d = {d}, ADF p-value = {p_value:.5f}")
        
        if p_value < significance:
            print(f"Series is stationary at d = {d}")
            return d
        # difference for next iteration
        diffed = diffed[1:] - diffed[:-1]
    
    print(f"Series did not become stationary after {max_d} differences")
    return max_d

In [34]:
find_d(data)

d = 0, ADF p-value = 0.94940
d = 1, ADF p-value = 0.00000
Series is stationary at d = 1


1

In [35]:
model = ARIMAX(data, p=1, d=1, q=1)
y_pred = model.forecast(steps=10)
full_y_pred = model.get_full(steps=10)

## Chọn p có RMSE thấp nhất

In [40]:
from sklearn.metrics import mean_squared_error

def find_p(data, max_p=10, d=1):
    best_p = 0
    min_mse = np.inf

    # Difference first
    if d > 0:
        diffed_data = ARIMAX(data, d=d).difference(data, d=d)
    else:
        diffed_data = np.array(data)

    for p in range(max_p + 1):
        model = ARIMAX(diffed_data, p=p, d=0, q=1) 
        _, ar_res = model.AR(diffed_data)

        # Truncate diffed_data to match ar_res length
        mse = mean_squared_error(diffed_data[-len(ar_res):], ar_res)
        print(mse)

        if mse < min_mse:
            min_mse = mse
            best_p = p

    return best_p

In [42]:
def find_q(data, p=1, max_q=10, d=1):
    best_q = 0
    min_mse = np.inf

    # 1. difference the data if d > 0
    if d > 0:
        diffed_data = ARIMAX(data, d=d).difference(data, d=d)
    else:
        diffed_data = np.array(data)

    # 2. loop over q values
    for q in range(max_q + 1):
        model = ARIMAX(diffed_data, p=p, d=0, q=q)  # d=0 because data already differenced

        # Fit AR first
        _, ar_res = model.AR(diffed_data)

        # Fit MA for current q
        _, ma_res = model.MA(diffed_data, ar_res)

        # Align lengths for MSE
        mse = mean_squared_error(diffed_data[-len(ma_res):], ma_res)

        if mse < min_mse:
            min_mse = mse
            best_q = q

    return best_q


In [43]:
find_q(data, p=0)

1